In [1]:
from utils_phoneme_reco import *


/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


# Wav2vec+CTC

In [2]:

MODEL_ID = "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wav2vec2-french-phonemizer"

model = AutoModelForCTC.from_pretrained(MODEL_ID)
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)

device = "cuda"
model = model.to(device)
model.eval()


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

# WavLM+CTC

In [2]:
import torch, librosa
from transformers import WavLMForCTC, Wav2Vec2FeatureExtractor, Wav2Vec2PhonemeCTCTokenizer

#CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme/checkpoint-268600"
CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme-large/checkpoint-476220"
ROOT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme-large"          # where vocab.json / tokenizer were saved

device = "cuda"
model = WavLMForCTC.from_pretrained(CKPT).to(device).eval()
feat  = Wav2Vec2FeatureExtractor.from_pretrained(CKPT)   # also present in CKPT
tok   = Wav2Vec2PhonemeCTCTokenizer.from_pretrained(ROOT)
tok.unk_token = "[UNK]"
tok.pad_token = "[PAD]"
model = model.to(device)
model.eval()

WavLMForCTC(
  (wavlm): WavLMModel(
    (feature_extractor): WavLMFeatureEncoder(
      (conv_layers): ModuleList(
        (0): WavLMLayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x WavLMLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x WavLMLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): WavLMFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
 

# Whisper+CTC

In [2]:
#whisper
from transformers import WhisperFeatureExtractor, Wav2Vec2PhonemeCTCTokenizer

CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/whisper-fr-phoneme/checkpoint-193924"
ROOT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/whisper-fr-phoneme"
device = "cuda"

model = WhisperEncoderForCTC.from_pretrained(CKPT).to(device).eval()
feat  = WhisperFeatureExtractor.from_pretrained(CKPT)
tok   = Wav2Vec2PhonemeCTCTokenizer.from_pretrained(ROOT)
tok.unk_token = "[UNK]"
tok.pad_token = "[PAD]"

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'WhisperTokenizer'. 
The class this function is called from is 'Wav2Vec2PhonemeCTCTokenizer'.


In [3]:
import os
import json
from pathlib import Path
import pandas as pd
import pickle

from jiwer import process_words
from collections import defaultdict
from pathlib import Path
from praatio import textgrid

audio_dir = "/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB"
textgrid_dir = Path("/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB")
ref_files = textgrid_dir.glob("*")
ref_dict = {
    str(f.stem)[:-11]: str(f)
    for f in ref_files
    if str(f.stem).endswith("pr_analyse")
}
patho = "cereb"

model_name = "whisper"

ref_inventory = set()
hyp_inventory = set()
alignment_store = {}

for filepath in textgrid_dir.glob("*"):
    f = filepath.stem + ".wav"
    audio_path = os.path.join(audio_dir, f)

    if os.path.exists(audio_path) and not filepath.stem.startswith("."):
        if '002710' in audio_path:
            continue

        textgrid_path = ref_dict[ filepath.stem]
        if model_name =="whisper":
            pred_phonemes, pred_alignments = get_phoneme_alignments_whisper_ctcfa(model, feat,tok, audio_path)
        elif model_name == "wavlm":
            pred_phonemes, pred_alignments = get_phoneme_alignments_wavlm_ctcfa(model, feat,tok, audio_path)
        else:
            pred_phonemes, pred_alignments = get_phoneme_alignments_w2v_ctcfa(model, processor, audio_path)

        ref_alignments = get_reference_alignments_typaloc(textgrid_path)

        clean_hyp = clean_alignment_dict(pred_alignments, is_hyp=True)
        clean_ref = clean_alignment_dict(ref_alignments,"typaloc",is_hyp=False)

        ref_seq = extract_phoneme_sequence(clean_ref)
        hyp_seq = extract_phoneme_sequence(clean_hyp)

        # collect inventories
        for ph in ref_seq:
            ref_inventory.add(ph)

        for ph in hyp_seq:
            hyp_inventory.add(ph)

        entry = {
            "file": f,
            "ref_intervals": clean_ref,
            "hyp_intervals": clean_hyp,
            "ref_seq": ref_seq,
            "hyp_seq": hyp_seq,
        }

        alignment_store[f] = entry



Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/vol/experiments3/imbenamor/TAPAS-FRAIS/src/utils/utils_phoneme_reco.py:120: UserWarning: torchaudio.functional._alignment.forced_align has been deprecated. This deprecation is part of a large re

Maximum timestamp in Textgrid changed from (112.198875) to (112.2)


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


Maximum timestamp in Textgrid changed from (94.87675) to (94.88)


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


...

Maximum timestamp in Textgrid changed from (127.059625) to (127.06)


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`
Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`
Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`
Lightning automatically upgraded your loaded checkpoint

Maximum timestamp in Textgrid changed from (94.87675) to (94.88)


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`
Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


Maximum timestamp in Textgrid changed from (112.198875) to (112.2)
Maximum timestamp in Textgrid changed from (127.059625) to (127.06)


In [4]:
# ==========================
# Inventory comparison
# ==========================

print(f"Reference inventory size: {len(ref_inventory)}")
print(f"Hypothesis inventory size: {len(hyp_inventory)}")

only_in_ref = sorted(ref_inventory - hyp_inventory)
only_in_hyp = sorted(hyp_inventory - ref_inventory)

if not only_in_ref and not only_in_hyp:
    print("\n✓ REF and HYP inventories are identical.")
else:
    print("\n✗ Inventories differ.")

    if only_in_ref:
        print("\nPhonemes present only in REF:")
        print(only_in_ref)

    if only_in_hyp:
        print("\nPhonemes present only in HYP:")
        print(only_in_hyp)

# Optional: print full inventories
print("\nREF inventory:")
print(sorted(ref_inventory))

print("\nHYP inventory:")
print(sorted(hyp_inventory))

Reference inventory size: 34
Hypothesis inventory size: 33

✗ Inventories differ.

Phonemes present only in REF:
['ɥ']

REF inventory:
['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɥ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']

HYP inventory:
['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']


In [5]:
len(alignment_store)

7

In [6]:
from metrics import *
import pickle
from jiwer import process_words
# (optional) save back so you don't redo it
with open(f"ctc_results/alignment_{model_name}_{patho}.pkl", "wb") as f:
    pickle.dump(alignment_store, f)


In [7]:
from metrics_alignment import *

with open(f"ctc_results/alignment_{model_name}_{patho}.pkl", "rb") as f:
    alignment_store = pickle.load(f)
compute_metrics(alignment_store, f"ctc_results/metrics++_{model_name}_{patho}.csv", per_phoneme_csv=None)


,group,style,K,AAS (ms),Median_start (ms),Mean_start (ms),Median_end (ms),Mean_end (ms),P90 (ms),%>50ms,...,Median_dur (ms),%dur>50ms,N_ref,N_hyp,PER (%),Deletions,Insertions,F1@0ms (%),F1@20ms (%),F1@50ms (%)
0,CCM-004773-01_L01.wav,ALL,458,112.649125,54.921086,91.464081,60.144939,133.834169,150.312500,59.497817,...,38.780973,37.554585,586,566,26.109215,45,25,0.0,23.784722,66.840278
1,CCM-004523-01_L01.wav,ALL,436,120.486136,54.062500,101.100712,56.663557,139.871560,117.301840,57.224771,...,30.028921,27.522936,562,548,28.469751,48,34,0.0,21.621622,66.306306
2,CCM-003998-01_L01.wav,ALL,426,116.322727,60.736654,81.351205,67.311116,151.294249,152.292363,67.253521,...,39.675240,40.845070,576,560,31.423611,47,31,0.0,19.542254,57.922535
3,CCM-004538-01_L01.wav,ALL,394,65.275216,52.945334,57.437359,52.830417,73.113074,104.490376,54.441624,...,25.323563,23.604061,580,535,36.896552,73,28,0.0,29.058296,71.928251
4,CCM-003094-01_L01.wav,ALL,445,109.030037,70.937500,85.647369,75.106825,132.412706,160.395381,74.157303,...,47.989155,46.067416,595,542,26.890756,63,10,0.0,17.062445,58.047493
5,CCM-003493-01_L01.wav,ALL,498,91.003135,58.437500,71.314208,65.322151,110.692063,152.763027,65.361446,...,40.945524,41.164659,632,614,27.215190,56,38,0.0,20.866774,60.353130
6,CCM-003110-01_L01.wav,ALL,457,81.117762,61.562500,69.072319,62.187500,93.163204,140.530604,66.630197,...,35.235787,37.199125,559,538,19.856887,30,9,0.0,18.778487,63.992707
7,STYLE_ALL,ALL,3114,99.648657,58.687788,79.784842,62.187500,119.512472,141.903800,63.680154,...,36.351453,36.576750,4090,3903,28.141809,362,175,0.0,21.518829,63.555611
8,GLOBAL,ALL,3114,99.648657,58.687788,79.784842,62.187500,119.512472,141.903800,63.680154,...,36.351453,36.576750,4090,3903,28.141809,362,175,0.0,21.518829,63.555611


In [8]:
dict_to_csv(alignment_store, f'ctc_results/pred_{model_name}_{patho}.csv')

# Prepare for MFA

In [9]:
df1=pd.read_csv(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/src/utils/ctc_results/pred_{model_name}_{patho}.csv")
df1["speaker_id"] =[i.split("_")[0] for i in df1["filename"]]
df1

,filename,predicted_phonemes,speaker_id
0,CCM-004773-01_L01.wav,d ɑ̃ z y n p ə t i v i l a ʒ d ə l a m ɔ̃ t a ...,CCM-004773-01
1,CCM-004523-01_L01.wav,d ɑ̃ z ɛ̃ p ə t i v i l a ʒ d ə l a m ɔ̃ t a ɲ...,CCM-004523-01
2,CCM-003998-01_L01.wav,d ɑ̃ z ɛ̃ p ə t i v i l a ʒ d ə l a m ɔ̃ t a ɡ...,CCM-003998-01
3,CCM-004538-01_L01.wav,d a l ɔ ʁ d ɑ̃ z ɛ̃ p ə t i v i l a ʒ d ə l a ...,CCM-004538-01
4,CCM-003094-01_L01.wav,d ɑ̃ z ɛ̃ p ə t i v i l a d ə l a m ɔ̃ t a ɲ l...,CCM-003094-01
5,CCM-003493-01_L01.wav,d ɑ̃ z ɛ̃ p ə t i v i l a ʒ d ə l a m ɔ̃ t a ɲ...,CCM-003493-01
6,CCM-003110-01_L01.wav,a d e o s ɔ̃ p a l e ʁ d ɑ̃ z ɛ̃ p ə t i v i l...,CCM-003110-01


In [10]:
df1["predicted_phonemes"] = df1["predicted_phonemes"].apply(normalize_phones)
ref_inventory = set()

for seq in df1["predicted_phonemes"].dropna():
    tokens = seq.split()
    ref_inventory.update(tokens)

print(sorted(ref_inventory))
print("Number of unique phonemes:", len(ref_inventory))

['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'œ', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']
Number of unique phonemes: 33


In [11]:
#Generation des fichiers pour mFA
import os
import torch
import torchaudio
import shutil

corpus_dir = f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/mfa_{patho}_{model_name}_ctc"
if os.path.exists(corpus_dir):
    shutil.rmtree(corpus_dir)  
os.makedirs(corpus_dir)
target_sr = 16000
for idx, row in df1.iterrows():
    audio_path = os.path.join(audio_dir, row["filename"])
    tokens = row["predicted_phonemes"]
    utt_id = os.path.splitext(os.path.basename(audio_path))[0]
    speaker_id = str(row["speaker_id"])  

    # Create speaker folder
    speaker_dir = os.path.join(corpus_dir, speaker_id)
    os.makedirs(speaker_dir, exist_ok=True)
    # Load audio
    waveform, sr = torchaudio.load(audio_path)
    # Resample if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        waveform = resampler(waveform)

    # Save wav inside speaker folder
    torchaudio.save(
        os.path.join(speaker_dir, f"{utt_id}.wav"),
        waveform,
        target_sr
    )

    # Save lab inside speaker folder
    with open(os.path.join(speaker_dir, f"{utt_id}.lab"), "w", encoding="utf-8") as f:
        f.write(tokens.strip())

/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch

In [12]:
with open(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/output_ph_reco/phoneme_{patho}_{model_name}_ctc.txt", "w", encoding="utf-8") as f:
    for ph in ref_inventory:
        f.write(f"{ph} {ph}\n")

# MFA ALIGN: GO to cmd 

# MFA alignment

In [9]:
#rhapsodie

import os
import json
from pathlib import Path
import pandas as pd
import pickle
from praatio import textgrid
import os
import json
from pathlib import Path
import pandas as pd
import pickle


tg_path = Path(f"/vol/experiments3/imbenamor/TAPAS-FRAIS/mfa_data/align_{model_name}_cereb")
ref_tg_path = Path("/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB")
audio_dir = "/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB"
ref_files = textgrid_dir.glob("*")
full_paths = list(tg_path.rglob("*.TextGrid"))
ref_dict = {
    str(f.stem)[:-11]: str(f)
    for f in ref_files
    if str(f.stem).endswith("pr_analyse")
}
alignment_store = {}
for hyp_path in full_paths:
    if ".ipynb_checkpoints" not in str(hyp_path):
        stem=hyp_path.stem
        #stem = hyp_path.stem.split("-")[1]+"-"+hyp_path.stem.split("-")[0] #txtgrid from mfa
        if stem not in ref_dict.keys():
            continue
    
        ref_path = ref_dict[stem]
        
        #f = hyp_path.stem.split("-")[1]+"-"+hyp_path.stem.split("-")[0] + ".wav"
        f = stem.split("-")[1]+"-"+stem.split("-")[0] + ".wav"
        audio_path = os.path.join(audio_dir, stem+".wav")
        if '002710' in audio_path:
            continue
        if os.path.exists(audio_path):
            phones_hyp, hyp_intervals = extract_phones_from_textgrid(hyp_path, t="phones")
            phones_ref, ref_intervals = extract_phones_from_textgrid_typaloc(ref_path)
            pred_alignments=[]
            for i,j,k in hyp_intervals:
                pred_alignments.append({"phoneme":k,"start":i,"end":j})
            ref_alignments=[]
            for i,j,k in ref_intervals:
                ref_alignments.append({"phoneme":k,"start":i,"end":j})
            clean_hyp = clean_alignment_dict(pred_alignments, is_hyp=True)
            clean_ref = clean_alignment_dict(ref_alignments, flag="typaloc", is_hyp=False)
            

            entry = {
                "file":          stem,
                "ref_intervals": clean_ref,
                "hyp_intervals": clean_hyp,
                "ref_seq":       extract_phoneme_sequence(clean_ref),
                "hyp_seq":       extract_phoneme_sequence(clean_hyp),
            }
            alignment_store[stem] = entry 

Maximum timestamp in Textgrid changed from (142.477688) to (142.48)
Maximum timestamp in Textgrid changed from (127.059625) to (127.06)
Maximum timestamp in Textgrid changed from (94.87675) to (94.88)
Maximum timestamp in Textgrid changed from (112.198875) to (112.2)


In [10]:
len(alignment_store)

7

In [11]:
from metrics import *
import pickle
from jiwer import process_words
# (optional) save back so you don't redo it
with open(f"mfa_results/alignment_{model_name}_mfa_{patho}.pkl", "wb") as f:
    pickle.dump(alignment_store, f)


In [12]:
from metrics_alignment import *

with open(f"mfa_results/alignment_{model_name}_mfa_{patho}.pkl", "rb") as f:
    alignment_store = pickle.load(f)
compute_metrics(alignment_store, f"mfa_results/metrics++_{model_name}_mfa_{patho}.csv", per_phoneme_csv=None)


,group,style,K,AAS (ms),Median_start (ms),Mean_start (ms),Median_end (ms),Mean_end (ms),P90 (ms),%>50ms,...,Median_dur (ms),%dur>50ms,N_ref,N_hyp,PER (%),Deletions,Insertions,F1@0ms (%),F1@20ms (%),F1@50ms (%)
0,CCM-003998-01_L01,ALL,426,64.546947,12.557722,65.115317,13.819269,63.978578,108.134678,20.774648,...,24.228331,30.046948,576,560,31.423611,47,31,1.760563,60.035211,77.112676
1,CCM-003094-01_L01,ALL,445,46.563587,10.004000,45.316027,10.002000,47.811147,99.998200,18.202247,...,23.141174,28.089888,595,542,26.890756,63,10,2.462621,67.018470,81.090589
2,CCM-003110-01_L01,ALL,457,29.452645,10.003000,30.508438,10.000000,28.396852,69.998800,13.457330,...,18.847933,22.975930,559,538,19.856887,30,9,2.370100,69.462170,85.323610
3,CCM-004538-01_L01,ALL,394,39.084017,10.001000,37.534856,10.000500,40.633178,59.593186,11.928934,...,19.500616,17.258883,580,535,36.896552,73,28,2.690583,69.596413,85.022422
4,CCM-004523-01_L01,ALL,436,84.931651,10.000000,83.281057,10.002000,86.582244,79.805465,13.761468,...,19.999000,21.788991,562,548,28.469751,48,34,3.603604,67.927928,83.423423
5,CCM-004773-01_L01,ALL,458,54.475788,11.381497,54.070450,11.606320,54.881126,88.466532,15.720524,...,19.512581,22.707424,586,566,26.109215,45,25,2.256944,67.361111,83.506944
6,CCM-003493-01_L01,ALL,498,32.649364,10.000000,32.808679,10.001000,32.490049,58.262580,11.445783,...,19.129224,20.281124,632,614,27.215190,56,38,1.605136,66.613162,82.343499
7,STYLE_ALL,ALL,3114,49.876767,10.003000,49.469934,10.084289,50.283599,79.178467,14.996789,...,20.000000,23.314066,4090,3903,28.141809,362,175,2.377080,66.833479,82.522207
8,GLOBAL,ALL,3114,49.876767,10.003000,49.469934,10.084289,50.283599,79.178467,14.996789,...,20.000000,23.314066,4090,3903,28.141809,362,175,2.377080,66.833479,82.522207
